# Grupo 7 - Examen Práctico RL

Integrantes: 

- Diego Valenzuela 22309
- Daniel Dubón 22233
- Joaquín Puente 22296
- Christian Echeverria XXXX
- Pendiente XXXX

In [2]:
import numpy as np
from collections import defaultdict

# ESPECIFICACIÓN DEL MDP
# Estado: (nivel_inventario, dias_hasta_vencimiento, demanda_promedio_7dias)
# nivel_inventario:       [0, 10, 20, ..., 100]              — 10 niveles
# dias_hasta_vencimiento: [1, 7, 14, 30, 60]                 — 5 niveles
# demanda_promedio_7dias: [bajo, medio, alto, crítico]        — 4 niveles
# Total: 200 estados | Acciones: [0, 10, 20, 30, 40, 50] unidades — 6 acciones

def transition(state, action):
    inventory, days_to_expiry, demand_level = state
    demand_map = {'bajo': 5, 'medio': 15, 'alto': 25, 'crítico': 40}
    daily_demand = demand_map[demand_level]
    new_inventory = min(100, max(0, inventory + action - daily_demand))
    new_days = max(1, days_to_expiry - 1)
    new_demand = demand_level
    return (new_inventory, new_days, new_demand)

def reward(state, action, next_state):
    new_inventory, new_days, _ = next_state
    inventory_reward = new_inventory * 0.5
    expiry_penalty = -10 if new_days <= 7 else 0
    order_penalty = -2 if action > 0 else 0
    return inventory_reward + expiry_penalty + order_penalty

def train(env, episodes=1000):
    Q = defaultdict(lambda: np.zeros(6))
    alpha = 0.9
    gamma = 0.99
    epsilon = 0.05
    for episode in range(episodes):
        state = env.reset()
        done = False
        while not done:
            if np.random.random() < epsilon:
                action = np.random.randint(6)
            else:
                action = np.argmax(Q[state])
            next_state, reward, done, _ = env.step(action)
            next_action = np.argmax(Q[next_state])
            td_target = reward + gamma * Q[next_state][next_action]
            Q[state][action] += alpha * (td_target - Q[state][action])
            state = next_state
    return Q

def preprocess_state(state):
    inventory, days_to_expiry, demand_level = state
    demand_map = {'bajo': 0, 'medio': 1, 'alto': 2, 'crítico': 3}
    features = np.array([
        inventory,
        days_to_expiry,
        demand_map[demand_level]
    ])
    return features

class LinearApproximator:
    def __init__(self, n_features=3, n_actions=6):
        self.weights = np.zeros((n_actions, n_features))
    def predict(self, features, action):
        return np.dot(self.weights[action], features)
    def update(self, features, action, target, alpha=0.01):
        prediction = self.predict(features, action)
        error = target - prediction
        self.weights[action] += alpha * error * features

def evaluate_policy(Q, env, episodes=100):
    total_rewards = []
    for episode in range(episodes):
        state = env.reset()
        episode_reward = 0
        done = False
        while not done:
            action = np.argmax(Q[state])
            next_state, reward, done, _ = env.step(action)
            episode_reward += reward
            state = next_state
        total_rewards.append(episode_reward)
    return {
        'mean_reward': np.mean(total_rewards),
        'std_reward': np.std(total_rewards),
        'min_reward': np.min(total_rewards)
    }

# MÉTRICAS DE ENTRENAMIENTO
# Episodio    Recompensa    Error TD    Política dominante
# 100         42.3          8.42        pedir_50
# 500         48.1          7.12        pedir_50
# 1000        51.7          5.21        pedir_50
# Varianza entre episodios: 0.8
# Política greedy: pedir 50 en 94% de estados

# RESULTADOS EN PRODUCCIÓN
resultados_produccion = {
    'stockouts_por_semana': 23,
    'productos_vencidos_por_semana': 41,
    'costo_almacenamiento_semanal': 8400,
    'costo_objetivo_semanal': 3200,
    'satisfaccion_cliente': 0.61
}

# RESULTADOS EN SIMULACIÓN
resultados_simulacion = {
    'mean_reward': 51.2,
    'std_reward': 0.9,
    'min_reward': 48.3
}

In [18]:
class FarmaciaEnv:
    def __init__(self):
        self.acciones_reales = [0, 10, 20, 30, 40, 50]
        self.state = None
        self.step_count = 0
        
    def reset(self):
        self.step_count = 0
        # EL FALLO ORIGINAL: El entorno inicia en un estado "normal".
        # Como transition() no cambia la demanda, los demás niveles jamás se visitan.
        self.state = (100, 30, 'medio') 
        return self.state
        
    def step(self, action_idx):
        self.step_count += 1
        action_unidades = self.acciones_reales[action_idx] 
        
        next_state = transition(self.state, action_unidades)
        r = reward(self.state, action_unidades, next_state)
        
        self.state = next_state
        done = self.step_count >= 7 
        return next_state, r, done, {}

entorno = FarmaciaEnv()
Q_entrenada = train(entorno, episodes=1000)

estados_visitados = len(Q_entrenada.keys())
estados_totales_posibles = 10 * 5 * 4
estados_no_visitados = estados_totales_posibles - estados_visitados

print(f"Total de estados posibles en el MDP: {estados_totales_posibles}")
print(f"Estados visitados por el agente de tu profe: {estados_visitados}")
print(f"Estados nunca visitados: {estados_no_visitados} ({(estados_no_visitados/estados_totales_posibles)*100:.1f}%)")

estado_produccion_critico = (0, 1, 'crítico') 

print("\n--- SIMULACIÓN DE FALLA EN PRODUCCIÓN ---")
print(f"Entra un cliente en producción. Estado actual: {estado_produccion_critico}")

if estado_produccion_critico not in Q_entrenada:
    print("Verificando en tabla Q: ¡Estado no encontrado!")
    valores_q = Q_entrenada[estado_produccion_critico]
    print(f"Valores Q internos devueltos: {valores_q}")
    
    accion_idx = np.argmax(valores_q)
    accion_unidades = entorno.acciones_reales[accion_idx]
    
    print(f"np.argmax selecciona el indice {accion_idx} que equivale a pedir {accion_unidades} unidades.")
    print("El sistema pide 0 unidades en plena crisis generando los stockouts.")
else:
    print("El estado fue visitado")

Total de estados posibles en el MDP: 200
Estados visitados por el agente de tu profe: 70
Estados nunca visitados: 130 (65.0%)

--- SIMULACIÓN DE FALLA EN PRODUCCIÓN ---
Entra un cliente en producción. Estado actual: (0, 1, 'crítico')
Verificando en tabla Q: ¡Estado no encontrado!
Valores Q internos devueltos: [0. 0. 0. 0. 0. 0.]
np.argmax selecciona el indice 0 que equivale a pedir 0 unidades.
El sistema pide 0 unidades en plena crisis generando los stockouts.


In [19]:
entorno = FarmaciaEnv()
Q_entrenada = train(entorno, episodes=1000)

estados_oficiales = []
for inv in range(0, 101, 10):
    for dias in [1, 7, 14, 30, 60]:
        for dem in ['bajo', 'medio', 'alto', 'crítico']:
            estados_oficiales.append((inv, dias, dem))

oficiales_visitados = [s for s in estados_oficiales if s in Q_entrenada]
oficiales_no_visitados = [s for s in estados_oficiales if s not in Q_entrenada]
estados_fantasma = len(Q_entrenada) - len(oficiales_visitados)

print(f"Total de estados creados en la tabla Q: {len(Q_entrenada)}")
print(f"Estados fantasma (generados por error en transition): {estados_fantasma}")
print(f"Estados OFICIALES visitados: {len(oficiales_visitados)} de 200")
print(f"Estados OFICIALES nunca visitados: {len(oficiales_no_visitados)} ({(len(oficiales_no_visitados)/200)*100:.1f}%)")

estado_produccion_sorpresa = oficiales_no_visitados[0]

print("\n--- SIMULACIÓN DE FALLA EN PRODUCCIÓN ---")
print(f"Entra un cliente en producción. Estado actual: {estado_produccion_sorpresa}")

if estado_produccion_sorpresa not in Q_entrenada:
    print("Verificando en tabla Q: ¡Estado no encontrado!")
    valores_q = Q_entrenada[estado_produccion_sorpresa]
    print(f"Valores Q internos devueltos: {valores_q}")
    
    accion_idx = np.argmax(valores_q)
    accion_unidades = entorno.acciones_reales[accion_idx]
    
    print(f"np.argmax selecciona el indice {accion_idx} que equivale a pedir {accion_unidades} unidades.")
    

Total de estados creados en la tabla Q: 77
Estados fantasma (generados por error en transition): 76
Estados OFICIALES visitados: 1 de 200
Estados OFICIALES nunca visitados: 219 (109.5%)

--- SIMULACIÓN DE FALLA EN PRODUCCIÓN ---
Entra un cliente en producción. Estado actual: (0, 1, 'bajo')
Verificando en tabla Q: ¡Estado no encontrado!
Valores Q internos devueltos: [0. 0. 0. 0. 0. 0.]
np.argmax selecciona el indice 0 que equivale a pedir 0 unidades.


### Entregable 7.1: 

Analicen las posibles causas del gap entre recompensa en simulación (51.2) y resultados en
producción (23 stockouts, costo 2.6x el objetivo). Para cada síntoma de producción, atribuyan la causa
probable a uno o más componentes del sistema usando el código como evidencia.

El gap entre la alta recompensa en simulación y el mal desempeño en producción pasa debido a fallas en el diseño del entorno y del algoritmo que hacen que el agente se confunda.

**1. Síntoma: Altos costos de almacenamiento (8400 vs 3200) y exceso de productos vencidos (41/semana)**
*   **Causa:** La función de recompensa está mal diseñada e incentiva el sobreabastecimiento constante.
*   **Evidencia en el código (`def reward`):** 
    *   `inventory_reward = new_inventory * 0.5`: El agente recibe una recompensa lineal positiva por acumular inventario donde mantener el inventario al máximo otorga 50 puntos en cada paso.
    *   `order_penalty = -2 if action > 0 else 0`: La penalización por pedir es estatica donde cuesta exactamente los mismos -2 puntos pedir 10 unidades que pedir 50.
*   **Componente responsable:** La función de recompensa (Grupo 2).
*   El agente aprendió que la política óptima matematica es pedir siempre la cantidad máxima de 50 para saturar el inventario y maximizar `inventory_reward` ignorando el impacto real del costo de bodega y los vencimientos.


**2. Síntoma: Alta cantidad de Stockouts (23/semana) y baja satisfacción del cliente**
*   **Causa:** El agente enfrenta situaciones en producción que nunca vio en entrenamiento y su comportamiento por defecto ante lo desconocido es no pedir nada.
*   **Evidencia en el código (`def transition` y `def train`):**
    *   `new_demand = demand_level`: En la función de transición la demanda es estatica durante todo el episodio. No hay transiciones dinamicas entre los niveles de demanda (bajo, medio, alto, critico).
    *   `epsilon = 0.05`: La tasa de exploración es demasiado baja para compensar la falta de variabilidad del entorno.
    *   `Q = defaultdict(lambda: np.zeros(6))`: Los estados no visitados se inicializan con valores Q de 0.
*   **Componentes responsables:** El MDP/Función de transición y La estrategia de exploración.
*   Debido a la demanda estatica y la baja exploración, el agente nunca visita estados con demanda "crítica" o fluctuante durante el entrenamiento. En producción, cuando la demanda salta y entra a un estado desconocido la tabla Q devuelve el arreglo inicial `[0., 0., 0., 0., 0., 0.]`. Al ejecutar `np.argmax(Q[state])`, la función selecciona por defecto el índice 0 (pedir 0 unidades) lo que garantiza un quiebre de stock en el momento de mayor demanda.

### Entregable 7.2: 

Analicen el comportamiento de la política greedy en estados no visitados durante el
entrenamiento. Argumenten cuántos de los 200 estados probablemente nunca fueron visitados, qué acción
toma el agente en esos estados, y cómo eso genera stockouts en producción.

El analisis exhaustivo de la tabla Q tras 1000 episodios revela una ceguera crítica en el agente, lo cual explica directamente los quiebres de stock en producción.

*   **Estados omitidos durante el entrenamiento:** Debido a que la función de transición original no introduce variabilidad en la demanda (`new_demand = demand_level`) y el entorno inicia en condiciones estándar el agente se aisla de la dinamica real del negocio y la simulación demostro que de los 200 estados validos del MDP, el agente omitió **199 estados (99.5%)**. Las situaciones críticas como inventarios bajos o demandas altas nunca fueron exploradas.
*   **Comportamiento de la politica greedy:** El código inicializa los valores de la tabla Q para estados desconocidos usando un arreglo de ceros: `Q = defaultdict(lambda: np.zeros(6))`. Cuando el agente, operando bajo una política puramente greedy en producción, se encuentra en un estado no visitado, la función evalúa el arreglo `[0. 0. 0. 0. 0. 0.]`.
*   **Acción resultante:** Ante un empate de valores máximos (todos son 0) la función `np.argmax()` devuelve por defecto el primer índice que es el `0`. En el mapeo de acciones del entorno, el índice 0 equivale a **pedir 0 unidades**.
*   **Impacto en producción (Stockouts):** En el entorno real de producción la demanda si fluctua. Cuando la cadena de farmacias entra en un estado crítico de alta demanda que el agente nunca vio en simulación, la política greedy se bloquea matemáticamente y decide no realizar pedidos (0 unidades). Esto genera sistemáticamente los 23 stockouts semanales, pues el agente deja de abastecer la farmacia exactamente en los momentos de mayor urgencia comercial.